# Notebook 04: Layer-wise SAE Sensitivity (Experiment 3)

Repeat PGD (fixed ε=0.1) across every layer that has a pretrained SAE.
Identifies which layers' SAE representations are most and least robust.

Connects to prior work on gated attention:
- In gated attention, early layers were most sensitive to perturbation
- Do early-layer SAEs show the same pattern?

Results → `results/exp3_layerwise_sensitivity.json`

> **Runtime note:** Loading a new SAE per layer is slow.  
> Default: 9 sampled layers. Set `LAYERS_TO_TEST = list(range(N_LAYERS))` for all.

In [ ]:
!pip install transformer-lens==1.19.0 sae-lens==4.3.1 transformers==4.44.2 \
    datasets==2.20.0 tqdm==4.66.4 scipy==1.13.1 -q

In [ ]:
import os, sys, json, random, time
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/rajo69/Adversarial-Robustness-of-SAE-Interpretations"
REPO_DIR = "/content/SAE_Adversarial_Project"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {GITHUB_REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"CWD: {os.getcwd()}")

In [ ]:
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR  = Path("figures");  FIGURES_DIR.mkdir(exist_ok=True)
print(f"Device: {DEVICE}")

In [ ]:
from transformer_lens import HookedTransformer

# Model only — SAE is loaded fresh per layer inside the loop to manage VRAM
try:
    model = HookedTransformer.from_pretrained("gemma-2-2b", device=DEVICE)
    MODEL_NAME   = "gemma-2-2b"
    SAE_RELEASE  = "gemma-scope-2b-pt-res"
    SAE_ID_TMPL  = "layer_{layer}/width_16k/average_l0_82"
    HOOK_TMPL    = "blocks.{layer}.hook_resid_post"
    N_LAYERS     = 26
    # Sample 9 layers; comment out for full run (slow on free T4)
    LAYERS_TO_TEST = [0, 3, 6, 9, 12, 15, 18, 21, 24]
except Exception as e:
    print(f"Gemma fallback ({e})")
    model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
    MODEL_NAME   = "gpt2"
    SAE_RELEASE  = "gpt2-small-res-jb"
    SAE_ID_TMPL  = "blocks.{layer}.hook_resid_pre"
    HOOK_TMPL    = "blocks.{layer}.hook_resid_pre"
    N_LAYERS     = 12
    LAYERS_TO_TEST = [0, 2, 4, 6, 8, 10, 11]

model.eval()
print(f"Model: {MODEL_NAME}  |  Layers to test: {LAYERS_TO_TEST}")
if DEVICE == "cuda": print(f"GPU after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
from src.data import load_eval_set, tokens_to_tensor

eval_set = load_eval_set()
# 50 prompts — loading a new SAE per layer is the bottleneck, not prompt count
torch.manual_seed(SEED)
idx = sorted(torch.randperm(len(eval_set))[:50].tolist())
exp_set = [eval_set[i] for i in idx]
print(f"Experiment subset: {len(exp_set)} prompts")

In [ ]:
EPSILON        = 0.1   # same fixed epsilon as the ε=0.1 condition in Exp 1
PGD_STEPS      = 20
N_RANDOM       = 5     # fewer random samples per prompt (speed)

print(f"Fixed ε={EPSILON}  |  PGD steps={PGD_STEPS}  |  Random samples={N_RANDOM}")
print(f"Total PGD runs: {len(LAYERS_TO_TEST)} layers × {len(exp_set)} prompts "
      f"= {len(LAYERS_TO_TEST)*len(exp_set)}")

In [ ]:
from sae_lens import SAE
from src.attacks import pgd_attack_sae, random_perturbation_baseline
from src.metrics import compute_all_metrics

layerwise = {}

for layer in tqdm(LAYERS_TO_TEST, desc="Layers"):
    t0 = time.time()
    sae_id = SAE_ID_TMPL.format(layer=layer)
    hn     = HOOK_TMPL.format(layer=layer)

    print(f"\nLayer {layer:2d} — loading {sae_id} …")
    try:
        sae, _, _ = SAE.from_pretrained(release=SAE_RELEASE, sae_id=sae_id)
        sae = sae.to(DEVICE); sae.eval()
    except Exception as exc:
        print(f"  SKIP — SAE unavailable: {exc}")
        continue

    adv_list, rnd_list = [], []

    for item in tqdm(exp_set, desc=f"L{layer}", leave=False):
        tok = tokens_to_tensor(item["tokens"], device=DEVICE)

        with torch.no_grad():
            cl_logits, cl_cache = model.run_with_cache(tok, names_filter=hn)
            cl_acts = sae.encode(cl_cache[hn])

        pert_emb = pgd_attack_sae(
            model=model, sae=sae, clean_tokens=tok,
            target_layer=layer, epsilon=EPSILON,
            steps=PGD_STEPS, device=DEVICE,
        )

        with torch.no_grad():
            def _h(v, h): return pert_emb
            pt_logits, pt_cache = model.run_with_cache(
                tok, names_filter=hn, fwd_hooks=[("hook_embed", _h)])
            pt_acts = sae.encode(pt_cache[hn])

        adv_list.append(compute_all_metrics(cl_acts, pt_acts, cl_logits, pt_logits))
        rnd_list.append(random_perturbation_baseline(
            model=model, sae=sae, clean_tokens=tok,
            target_layer=layer, epsilon=EPSILON,
            n_samples=N_RANDOM, seed=SEED, device=DEVICE,
        ))
        if DEVICE == "cuda": torch.cuda.empty_cache()

    # Free SAE VRAM before loading next layer
    del sae
    if DEVICE == "cuda": torch.cuda.empty_cache()

    ac = [m["cosine_similarity"]   for m in adv_list]
    rc = [m["mean_cosine"]         for m in rnd_list]
    af = [m["feature_flip_count"]  for m in adv_list]
    aj = [m["jaccard_index"]       for m in adv_list]
    adv_adv = [(1-a)/(1-r+1e-10) for a, r in zip(ac, rc)]

    layerwise[str(layer)] = {
        "layer": layer,
        "adversarial": adv_list,
        "random": rnd_list,
        "summary": {
            "adv_cosine_mean":  float(np.mean(ac)), "adv_cosine_std":  float(np.std(ac)),
            "rnd_cosine_mean":  float(np.mean(rc)),
            "adv_jaccard_mean": float(np.mean(aj)),
            "adv_flip_mean":    float(np.mean(af)),
            "adv_advantage":    float(np.mean(adv_adv)),
            "runtime_s":        float(time.time()-t0),
        },
    }
    s = layerwise[str(layer)]["summary"]
    print(f"  L{layer:2d}: adv_cos={s['adv_cosine_mean']:.4f}  "
          f"rnd_cos={s['rnd_cosine_mean']:.4f}  "
          f"advantage={s['adv_advantage']:.2f}×  "
          f"time={s['runtime_s']:.0f}s")

In [ ]:
output = {
    "experiment": "layerwise_sensitivity", "model": MODEL_NAME,
    "sae_release": SAE_RELEASE, "layers_tested": LAYERS_TO_TEST,
    "epsilon": EPSILON, "pgd_steps": PGD_STEPS,
    "n_prompts": len(exp_set), "seed": SEED,
    "per_layer": layerwise,
}
p = RESULTS_DIR / "exp3_layerwise_sensitivity.json"
p.write_text(json.dumps(output, indent=2))
print(f"Saved: {p}")

In [ ]:
from src.plot_config import apply_style, save_fig, COLORS, LAYER_CMAP

apply_style()
layers_done = [int(k) for k in layerwise]
ac_m = [layerwise[str(l)]["summary"]["adv_cosine_mean"] for l in layers_done]
ac_s = [layerwise[str(l)]["summary"]["adv_cosine_std"]  for l in layers_done]
rc_m = [layerwise[str(l)]["summary"]["rnd_cosine_mean"] for l in layers_done]
af_m = [layerwise[str(l)]["summary"]["adv_flip_mean"]   for l in layers_done]
aa_m = [layerwise[str(l)]["summary"]["adv_advantage"]   for l in layers_done]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    f"Exp 3: Layer-wise SAE Robustness\n{MODEL_NAME}, ε={EPSILON}", fontsize=13)

# 1) Cosine similarity by layer
ax = axes[0]
ax.errorbar(layers_done, ac_m, yerr=ac_s, fmt='o-',
            color=COLORS["adversarial"], capsize=4, label="PGD")
ax.plot(layers_done, rc_m, 's--', color=COLORS["random"], alpha=0.7, label="Random")
ax.set_xlabel("Layer"); ax.set_ylabel("SAE Cosine Similarity")
ax.set_title("SAE Robustness by Layer\n(↑ = more robust)")
ax.legend()

# 2) Feature flip count by layer
ax = axes[1]
cmap = plt.cm.get_cmap(LAYER_CMAP)
cols = cmap(np.linspace(0.2, 0.85, len(layers_done)))
ax.bar(range(len(layers_done)), af_m, color=cols)
ax.set_xticks(range(len(layers_done)))
ax.set_xticklabels([str(l) for l in layers_done])
ax.set_xlabel("Layer"); ax.set_ylabel("Mean Feature Flip Count")
ax.set_title("Feature Flips by Layer\n(↑ = less stable)")

# 3) Adversarial advantage by layer
ax = axes[2]
ax.bar(range(len(layers_done)), aa_m, color=COLORS["adversarial"], alpha=0.8)
ax.axhline(1.0, color="black", ls="--", lw=1.2, label="Random baseline (1×)")
ax.set_xticks(range(len(layers_done)))
ax.set_xticklabels([str(l) for l in layers_done])
ax.set_xlabel("Layer"); ax.set_ylabel("Adversarial Advantage")
ax.set_title("PGD vs Random Disruption\n(>1 = genuine adversarial vulnerability)")
ax.legend()

plt.tight_layout()
fp = FIGURES_DIR / "exp3_layerwise_sensitivity.png"
save_fig(fig, str(fp)); plt.show()
print(f"Saved: {fp}")

## ✓ All experiments complete

**Record in `progress.md`:** most/least robust layers, whether early layers mirror
gated-attention sensitivity, adversarial advantage values.

### Interpretation guide
- **Early layers more fragile than late layers** → mirrors gated-attention finding;
  suggests a structural property of the model architecture.
- **Late layers more fragile** → SAE fragility and gated-attention sensitivity
  may be measuring different things.
- **Adversarial advantage ≈ 1 at all layers** → vulnerability is mostly noise
  sensitivity, not exploitable adversarial structure.
- **Adversarial advantage >> 1** → genuine exploitable vulnerability; an adversary
  can do much better than random noise.

### Download results from Colab
```python
from google.colab import files
import glob
for f in glob.glob('results/*.json'):
    files.download(f)
for f in glob.glob('figures/*.png'):
    files.download(f)
```

Then bring JSONs back to your local `results/` and run the analyst agent.